<a href="https://colab.research.google.com/github/Badar063/-UberRide-Optimizer---Smart-Ride-Management-System/blob/main/Deep_Dive_Note_Taker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install OpenAI Whisper for converting speech to text
!pip install -q openai-whisper

# Install Sentence Transformers for creating the RAG memory (embeddings)
!pip install -q sentence-transformers

# Install Google Generative AI to use the Gemini LLM
!pip install -q google-generativeai

# Install ffmpeg which is required for audio processing
!apt-get install -y ffmpeg

# Import the operating system library to manage files
import os

# Import the Google Generative AI library
import google.generativeai as genai

# Import Whisper for speech recognition
import whisper

# Import the sentence transformer for RAG capabilities
from sentence_transformers import SentenceTransformer, util

# SET YOUR GOOGLE API KEY HERE
# Replace 'YOUR_API_KEY' with your actual key from Google AI Studio
my_api_key = "Add Your API Here"

# Configure the Gemini library with your API key
genai.configure(api_key=my_api_key)

# Initialize the Gemini model (using Gemini Pro for text)
llm_model = genai.GenerativeModel('gemini-pro')

# Initialize the Whisper model (using 'base' for speed and decent accuracy)
# We specify 'cuda' to use the GPU for faster processing
stt_model = whisper.load_model("base", device="cuda")

# Initialize the embedding model for RAG (retrieval)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Import the file upload widget specifically for Google Colab
from google.colab import files

# Prompt the user to upload an audio file
print("Please upload your audio file (mp3 or wav):")

# Execute the upload function
uploaded = files.upload()

# Get the name of the file you just uploaded
audio_filename = list(uploaded.keys())[0]

# Print a status message indicating transcription has started
print("Transcribing audio... this may take a moment.")

# Transcribe the audio file using Whisper
# This converts speech into a raw text string
result = stt_model.transcribe(audio_filename)

# Extract the text portion from the result
full_text = result['text']

# Print the first 200 characters to verify it worked
print(f"Transcription Preview: {full_text[:200]}...")

# --- STEP: SUMMARIZATION ---

# Create a prompt for the LLM to summarize the text
summary_prompt = f"Summarize the following text into bullet points:\n{full_text}"

# Send the prompt to Gemini to get the summary
summary_response = llm_model.generate_content(summary_prompt)

# Print the summary heading
print("\n--- SUMMARY ---")

# Print the actual text of the summary
print(summary_response.text)

# --- STEP: RAG (RETRIEVAL AUGMENTED GENERATION) ---

# Split the full text into smaller chunks (sentences or segments)
# This allows us to search for specific parts later
chunks = full_text.split('. ')

# Convert these text chunks into numerical vectors (embeddings)
embeddings = embed_model.encode(chunks, convert_to_tensor=True)

# Define a loop to ask questions endlessly until typed 'exit'
while True:
    # Ask the user for a question
    user_query = input("\nAsk a question about the audio (or type 'exit'): ")

    # Check if the user wants to quit
    if user_query.lower() == 'exit':
        # Break the loop to stop the program
        break

    # Convert the user's question into a numerical vector
    query_embedding = embed_model.encode(user_query, convert_to_tensor=True)

    # Search for the most relevant chunk in our data using semantic search
    # This finds the piece of text most similar to the question
    hits = util.semantic_search(query_embedding, embeddings, top_k=1)

    # Get the index (location) of the best matching chunk
    best_hit_index = hits[0][0]['corpus_id']

    # Retrieve the actual text of that best matching chunk
    relevant_context = chunks[best_hit_index]

    # Create a prompt combining the context and the question
    rag_prompt = f"Context: {relevant_context}\n\nQuestion: {user_query}\nAnswer based on context:"

    # Ask Gemini to answer using that specific context
    answer = llm_model.generate_content(rag_prompt)

    # Print the answer
    print(f"Answer: {answer.text}")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Please upload your audio file (mp3 or wav):


Saving the-psychology-of-narcissism-w-keith-campbell-128-ytshorts.savetube.me.mp3 to the-psychology-of-narcissism-w-keith-campbell-128-ytshorts.savetube.me (1).mp3
Transcribing audio... this may take a moment.
Transcription Preview:  The Ancient Greeks and Romans had a myth about someone a little too obsessed with his own image. In one telling, Narcissus was a handsome guy wandering the world in search of someone to love. After r...


NotFound: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

In [ ]:
# Define the filename for the transcript
transcript_filename = "audio_transcript.txt"

# Write the full_text to the file
with open(transcript_filename, "w") as f:
    f.write(full_text)

# Create a download link for the file
from IPython.display import FileLink
display(FileLink(transcript_filename))

/content/audio_transcript.txt